In [1]:
import re
import gensim
import nltk
from nltk.corpus import stopwords
import spacy
import gensim.corpora as corpora
from gensim.models import CoherenceModel
from gensim.models.callbacks import PerplexityMetric, ConvergenceMetric, CoherenceMetric
import logging
import os
import json
import pandas as pd

# Latent Dirichlet Allocation on Abstracts

In [2]:
# preprocessing

def preprocess_text(text):
    text = re.sub('\s+', ' ', text)  # Remove extra spaces
    text = re.sub('\S*@\S*\s?', '', text)  # Remove emails
    text = re.sub('\'', '', text)  # Remove apostrophes
    text = re.sub('[^a-zA-Z]', ' ', text)  # Remove non-alphabet characters
    text = text.lower()  # Convert to lowercase
    return text

# Download NLTK stopwords
nltk.download('stopwords')
stop_words = stopwords.words('english')

# Tokenize and remove stopwords
def tokenize(text):
    tokens = gensim.utils.simple_preprocess(text, deacc=True)
    tokens = [token for token in tokens if token not in stop_words]
    return tokens

# Load spaCy model
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

def lemmatize(tokens):
    doc = nlp(" ".join(tokens))
    return [token.lemma_ for token in doc]

[nltk_data] Downloading package stopwords to /Users/ruth/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
# load all abstracts into a dataframe

abstracts = []
file = []
titles = []
is_quantitative = []
is_code_publicly_available = []
is_data_repository_available = []

# load the 10k data
fla_10k = pd.read_csv("fla_csvs/cites.csv")


meta_dir = "meta"
for fname in os.listdir(meta_dir):
    if fname.endswith(".json"):
        fpath = os.path.join(meta_dir, fname)
        with open(fpath, "r") as f:
            try:
                data = json.load(f)
                if data.get("article_subtype", "").lower() in ("fla"):
                    # load is_quantiative, is_code_publicly_available, is_data_publicly_available from fla_10k
                    doi = data.get("doi", "")
                    is_quantitative.append(fla_10k.loc[fla_10k["doi"] == doi, "is_quantitative_study"].values[0] if doi in fla_10k["doi"].values else None)
                    is_code_publicly_available.append(fla_10k.loc[fla_10k["doi"] == doi, "is_code_publicly_available"].values[0] if doi in fla_10k["doi"].values else None)
                    is_data_repository_available.append(fla_10k.loc[fla_10k["doi"] == doi, "is_data_repository_available"].values[0] if doi in fla_10k["doi"].values else None)

                    abstract = data.get("title") + " " + data.get("abstract") + " ".join(data.get("keywords")) + " " + data.get("journal")
                    abstracts.append(abstract)
                    file.append(data.get("doi", ""))
                    titles.append(data.get("title", ""))
            except Exception as e:
                print(f"Error reading {fname}: {e}")

print(len(abstracts), "abstracts loaded.")

# make the abstracts list and file list into a dataframe

data = {
    'abstract': abstracts,
    'doi': file,
    'title': titles,
    'is_data_repository_available': is_data_repository_available,
    'is_code_publicly_available': is_code_publicly_available,
    'is_quantitative': is_quantitative
}
df = pd.DataFrame(data)
df_sample = df

df_sample.head()

5 abstracts loaded.


,abstract,doi,title,is_data_repository_available,is_code_publicly_available,is_quantitative
0,Barriers to investing in cycling: Stakeholder ...,10.1016/j.tra.2017.11.003,Barriers to investing in cycling: Stakeholder ...,False,False,True
1,Elevating access: Comparing accessibility to j...,10.1016/j.tra.2018.02.017,Elevating access: Comparing accessibility to j...,False,False,True
2,Can product service systems support electric v...,10.1016/j.tra.2018.04.028,Can product service systems support electric v...,False,False,True
3,Taking responsibility: A responsible research ...,10.1016/j.tra.2018.05.004,Taking responsibility: A responsible research ...,False,False,False
4,Impacts of an active travel intervention with ...,10.1016/j.tra.2018.05.018,Impacts of an active travel intervention with ...,False,False,True


In [4]:
df_sample["cleaned"] = df_sample["abstract"].apply(preprocess_text)
print("Cleaned abstracts.")
df_sample["tokens"] = df_sample["cleaned"].apply(tokenize)
print("Tokenized abstracts.")
df_sample["lemmas"] = df_sample["tokens"].apply(lemmatize)

model_path = 'lda_data/my_lda_model'
lda_model = gensim.models.LdaModel.load(model_path)
id2word = lda_model.id2word

# id2word = corpora.Dictionary(df_sample["lemmas"])
# id2word.filter_extremes(no_below=20, no_above=0.8)
texts = df_sample["lemmas"]
corpus = [id2word.doc2bow(text) for text in texts]

df_sample.head()

Cleaned abstracts.
Tokenized abstracts.


,abstract,doi,title,is_data_repository_available,is_code_publicly_available,is_quantitative,cleaned,tokens,lemmas
0,Barriers to investing in cycling: Stakeholder ...,10.1016/j.tra.2017.11.003,Barriers to investing in cycling: Stakeholder ...,False,False,True,barriers to investing in cycling stakeholder ...,"[barriers, investing, cycling, stakeholder, vi...","[barrier, invest, cycling, stakeholder, view, ..."
1,Elevating access: Comparing accessibility to j...,10.1016/j.tra.2018.02.017,Elevating access: Comparing accessibility to j...,False,False,True,elevating access comparing accessibility to j...,"[elevating, access, comparing, accessibility, ...","[elevate, access, compare, accessibility, job,..."
2,Can product service systems support electric v...,10.1016/j.tra.2018.04.028,Can product service systems support electric v...,False,False,True,can product service systems support electric v...,"[product, service, systems, support, electric,...","[product, service, system, support, electric, ..."
3,Taking responsibility: A responsible research ...,10.1016/j.tra.2018.05.004,Taking responsibility: A responsible research ...,False,False,False,taking responsibility a responsible research ...,"[taking, responsibility, responsible, research...","[take, responsibility, responsible, research, ..."
4,Impacts of an active travel intervention with ...,10.1016/j.tra.2018.05.018,Impacts of an active travel intervention with ...,False,False,True,impacts of an active travel intervention with ...,"[impacts, active, travel, intervention, cyclin...","[impact, active, travel, intervention, cycling..."


In [5]:
len(id2word)  # Number of unique tokens in the dictionary
len(corpus)

5

# Train LDA from scratch - skip if not training from scratch

In [13]:
for i in range(15, 16):
    # turn on logging in gensim
    lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus,
                                                id2word=id2word,
                                                num_topics=i, 
                                                random_state=100,
                                                update_every=1,
                                                chunksize=1000,
                                                iterations=200,
                                                passes=50,
                                                alpha='auto',
                                                per_word_topics=True)
    
    coherence_model_lda = CoherenceModel(model=lda_model, texts=df_sample['lemmas'], dictionary=id2word, coherence='c_v')
    coherence_lda = coherence_model_lda.get_coherence()
    print('\nCoherence Score: ', coherence_lda)



Coherence Score:  0.4994091738809963


In [82]:
topics = lda_model.print_topics(num_words=15)
for topic in topics:
    print(topic)

topics = {
    0: "Road Infrastructure & Emergency Management",
    1: "COVID-19 Impact on Travel & Activities",
    2: "Driver Behavior & Safety Risk",
    3: "Social & Policy Aspects of Mobility",
    4: "Supply Chain & Market Strategies",
    5: "Traffic Flow & Network Control",
    6: "Automated Driving & Human Factors",
    7: "Optimization & Routing Algorithms",
    8: "Travel & Mode Choice Behavior",
    9: "Public Transit Service & Demand",
    10: "Transportation Emissions & Policy",
    11: "Urban Environment & Active Transport",
    12: "Data-Driven Modeling & Prediction",
    13: "Electric Vehicles & Ride-Sharing",
    14: "Air & Freight Logistics"
}

(0, '0.093*"speed" + 0.074*"road" + 0.020*"evacuation" + 0.017*"cluster" + 0.017*"limit" + 0.015*"zone" + 0.015*"highway" + 0.015*"traffic" + 0.014*"graph" + 0.013*"disaster" + 0.012*"sign" + 0.011*"pavement" + 0.010*"emergency" + 0.009*"high" + 0.009*"area"')
(1, '0.027*"time" + 0.018*"covid" + 0.016*"travel" + 0.015*"activity" + 0.014*"pandemic" + 0.013*"work" + 0.012*"change" + 0.012*"impact" + 0.011*"day" + 0.010*"use" + 0.009*"long" + 0.009*"increase" + 0.008*"effect" + 0.008*"noise" + 0.008*"study"')
(2, '0.033*"driver" + 0.025*"behaviour" + 0.023*"drive" + 0.022*"risk" + 0.022*"safety" + 0.019*"traffic" + 0.016*"factor" + 0.015*"use" + 0.015*"study" + 0.015*"crash" + 0.013*"perceive" + 0.013*"behavior" + 0.013*"perception" + 0.013*"road" + 0.009*"psychology"')
(3, '0.014*"study" + 0.014*"policy" + 0.012*"mobility" + 0.011*"social" + 0.011*"use" + 0.010*"transport" + 0.009*"public" + 0.009*"technology" + 0.008*"analysis" + 0.008*"perspective" + 0.008*"intention" + 0.007*"system" 

In [ ]:
model_path = "my_new_lda_model"

# Save the model
lda_model.save(model_path)
print(f"Gensim LDA model saved to {model_path}")

Gensim LDA model saved to my_lda_model


# Load LDA from memory

In [6]:
# load LDA from my_lda_model
model_path = 'lda_data/my_lda_model'
lda_model = gensim.models.LdaModel.load(model_path)
id2word = lda_model.id2word

# Code to generate data for tsne plot in tsne.ipynb

In [16]:
topics = {
    0: "Resilience and evacuation",
    1: "COVID-19 and pandemic",
    2: "Safety aspects of driving behavior",
    3: "Mobility policy",
    4: "Supply Chain",
    5: "Traffic Flow and Control",
    6: "Driving behavior in mixed autonomy",
    7: "Optimization",
    8: "Choice Model",
    9: "Public Transit",
    10: "Emissions",
    11: "Active Transportion and land use",
    12: "Data-Driven Modeling and Prediction",
    13: "Electric Vehicles and Ride-Sharing",
    14: "Air and Freight Logistics"
}

fla_10k["lda_topic"] = None

for index, row in df_sample.iterrows():
    # print(row)
    
    doc_bow = id2word.doc2bow(row['lemmas'])
    topics_for_doc = lda_model.get_document_topics(doc_bow, minimum_probability=-0.1)
    top_topic = max(topics_for_doc, key=lambda x: x[1])[0]

    fla_10k.loc[index, "lda_topic"] = topics[top_topic]

In [17]:
fla_10k.to_csv("fla_csvs/lda.csv")